# 🧠 Deep Dive: Databricks, Apache Spark, and Delta Lake

In this notebook, we explore why big data architectures don't use PostgreSQL or standard CSV files for feature engineering and quantitative analysis.

### Core Concepts You Will Learn:
1. **Vertical Scaling (Postgres) vs Horizontal Scaling (Spark)**
2. **What makes Apache Spark so powerful?**
3. **Data Lakes vs Delta Lakes**: Fixing the "Corrupted File" problem with ACID transactions.

In [1]:
# Setup dependencies (PySpark requires JVM internally, but can run locally in Python)
!pip install pyspark pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 68.7 MB/s  0:00:06:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008706 sha256=3d9c35215412e3d625605c26bab217132beca43ffb967717d8111cf62c480299
  Stored in directory: /Users/ryoichi/Library/Caches/pip/wheels/4f/8d/4c/277e6dd4fb378bc5528039d8db36e6b57440622db692fe9b51
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


## 1. Relational Databases (PostgreSQL) vs Apache Spark

### 💡 The Problem with PostgreSQL
PostgreSQL is a massive single computer (Vertical Scaling). If you want to compute the 25-day Moving Average of every stock in the S&P 500 spanning 20 years, an RDBMS has to scan rows sequentially on one database server's single hard drive. Eventually, it runs out of RAM or CPU and grinds to a halt.

### ✅ The Solution: Databricks / Apache Spark
Databricks orchestrates **Apache Spark**. Spark splits your dataset into a thousand smaller chunks across hundreds of separate computers (Horizontal Scaling). It runs the Moving Average math simultaneously in RAM across all nodes, and stitches the answer back together in seconds.

Let's execute actual PySpark code to see how it looks identical to standard code, but operates on distributed architecture!

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg

# Initialize a local Spark "Cluster" inside this notebook
print("Spinning up local Apache Spark cluster...")
spark = SparkSession.builder.appName("TradingBacktestApp").getOrCreate()

# Create dummy stock data spreading across our DataFrame
data = [
    ("7203.T", "2024-01-01", 2000.0),
    ("7203.T", "2024-01-02", 2050.0),
    ("7203.T", "2024-01-03", 2100.0),
    ("7203.T", "2024-01-04", 1950.0),
    ("7203.T", "2024-01-05", 2010.0),
]
df_spark = spark.createDataFrame(data, ["Ticker", "Date", "Close"])

print("\nRaw Spark Distributed DataFrame:")
df_spark.show()

Spinning up local Apache Spark cluster...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/20 17:02:58 WARN Utils: Your hostname, naitouryouichinoMacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.101.155 instead (on interface en0)
26/02/20 17:02:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/20 17:03:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/20 17:03:05 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:582)
	at java.base/java.util.Servic


Raw Spark Distributed DataFrame:


+------+----------+------+
|Ticker|      Date| Close|
+------+----------+------+
|7203.T|2024-01-01|2000.0|
|7203.T|2024-01-02|2050.0|
|7203.T|2024-01-03|2100.0|
|7203.T|2024-01-04|1950.0|
|7203.T|2024-01-05|2010.0|
+------+----------+------+



### Calculating Distributed Features
Notice how we define a "Window". In Spark, data is scattered across machines. A `Window.partitionBy("Ticker")` tells Spark to ensure all "7203.T" stock records get sent to the exact same computing node, so the moving average is calculated correctly without dragging data across the network.

In [3]:
# Distribute the data by Ticker, sort by Date
windowSpec = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-2, 0) # 3-day MA for simplicity

# Calculate average across the window
df_features = df_spark.withColumn("SMA_3", avg(col("Close")).over(windowSpec))

print("Feature Engineered Distributed DataFrame:")
df_features.show()

Feature Engineered Distributed DataFrame:
+------+----------+------+------------------+
|Ticker|      Date| Close|             SMA_3|
+------+----------+------+------------------+
|7203.T|2024-01-01|2000.0|            2000.0|
|7203.T|2024-01-02|2050.0|            2025.0|
|7203.T|2024-01-03|2100.0|            2050.0|
|7203.T|2024-01-04|1950.0|2033.3333333333333|
|7203.T|2024-01-05|2010.0|            2020.0|
+------+----------+------+------------------+



## 2. Standard Data Lakes (CSV/Parquet) vs Delta Lake

So far, we have high performance computation using Spark. Now we need to save the data permanently to storage like AWS S3 or Azure Blob.

### ❌ The "Standard CSV" Problem
In a normal Data Lake, you save a giant `.csv` or `.parquet` file. But what if the cluster crashes right in the middle of writing the CSV?
You are left with half a file. The next person who reads the data gets corrupted, broken CSV data, and a crashed application.

Let's simulate a corrupted write.

In [5]:
import os
import time

filename_csv = "corrupt_data.csv"

def simulate_crashing_csv_write():
    print("Writing 1,000,000 rows to CSV...")
    with open(filename_csv, "w") as f:
        f.write("Ticker,Date,Close\n")
        f.write("7203.T,2024-01-01,2000.0\n")
        f.write("7203.T,2024-01-02,2050.0\n")
        # --- CRASH SIMULATION ---
        print("💥 CRITICAL SERVER ERROR: Out of Memory! Crash!")
        f.write("7203.T,2") # File cuts off mid-write!

simulate_crashing_csv_write()

print("\nTrying to read the corrupted CSV:")
with open(filename_csv, "r") as f:
    print(f.read())

Writing 1,000,000 rows to CSV...
💥 CRITICAL SERVER ERROR: Out of Memory! Crash!

Trying to read the corrupted CSV:
Ticker,Date,Close
7203.T,2024-01-01,2000.0
7203.T,2024-01-02,2050.0
7203.T,2


### ✅ The Solution: Delta Lake ACID Transactions

Databricks invented **Delta Lake**. Instead of just writing a file, it writes into a highly structured transaction log.

If Delta Lake crashes mid-write, it completely rolls back the failed transaction, acting exactly as a mature Postgres database would. The user trying to read the data will simply see the old, clean version of the data until the new data safely commits 100%.

Furthermore, Delta Lake enables **Time Travel**. Because it logs everything, you can literally run SQL like:
`SELECT * FROM my_table TIMESTAMP AS OF '2023-01-01'` and get the exact snapshot of data that existed on that day! This is critical for ML models to prevent look-ahead bias and rebuild old models.

### 🎙️ The Interview Defense
**Interviewer:** "Why use Databricks Delta Lake instead of saving to a Postgres RDS database?"

**Your Answer:** "Postgres scales vertically. It is fundamentally impossible to calculate 20-year rolling window functions across 5,000 stocks on a single database efficiently. I used Databricks because Apache Spark scales horizontally across hundreds of nodes to slice feature engineering times from hours to seconds. However, traditional Data Lakes (S3 buckets of Parquet files) lack ACID transactions, meaning pipeline crashes corrupt data. Delta Lake bridges this gap, providing horizontal massive-scale computing paired with the safety of atomic rollback transactions and Time Travel versioning." 